# Compare cost only after models pass quality gates

This credential-free lab adapts MLflow's [cost-quality trade-off cookbook](https://mlflow.org/cookbook/cost-quality-tradeoff/) to logical platform resources and governed evidence.

The cookbook's durable idea is to hold the prompt, cases, scorers, and inference settings fixed while changing only the model. This repository does not embed vendor model IDs or a price table in application code. Use trace-recorded cost when available and the platform gateway or billing system as the chargeback source of truth.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Freeze the comparison contract

A model comparison is interpretable only when every other material input is identical. Evaluation-judge cost is reported separately from target-inference cost so a more expensive rubric cannot masquerade as a more expensive application model.

In [ ]:
from examples.support.cost_quality import comparison_contract

comparison_contract()

## 2. Run the fixture models, then inspect quality, tokens, cost, and coverage together

Four deterministic offline fixture models actually run over the evaluation records: `draft-chat` copies the excerpt but appends prohibited advice, `economy-chat` answers tersely with one citation, `general-chat` repeats the excerpt verbosely, and `quality-chat` answers best of all but reports no usage. Every quality score, token count, and cost below is computed from the answer strings and a simulated non-vendor price card, and stays labelled `simulated_offline_fixture` because it demonstrates decision shape, not provider performance or current pricing. `quality-chat`'s missing cost is a consequence of its missing usage evidence, not a typed-in value.

In [ ]:
from examples.support.cost_quality import build_cost_comparison

comparison = build_cost_comparison()
shown = comparison[
    [
        "logical_model",
        "quality_score",
        "critical_case_pass_rate",
        "total_tokens",
        "latency_ms_mean",
        "target_inference_cost_usd",
        "evaluation_judge_cost_usd",
        "cost_coverage",
    ]
]
print("Every row is computed; measurement_source stays 'simulated_offline_fixture'.")
print(shown.to_string(index=False))

## 3. Filter by release quality before ranking cost

Quality per dollar is a decision aid, never a release gate. Missing cost remains unknown. A model with incomplete cost coverage can be high quality but cannot win a cost comparison.

In [ ]:
from examples.support.cost_quality import cost_ranking, quality_gate_report

print("Quality gate, before any cost is discussed:")
print(quality_gate_report(comparison).to_string(index=False))
print()
print("Cost ranking among cost-comparable survivors:")
print(cost_ranking(comparison).to_string(index=False))

In [ ]:
from examples.support.cost_quality import comparison_decision

cost_decision = comparison_decision(comparison)
for key, value in cost_decision.items():
    print(f"{key}: {value}")
excluded = ", ".join(cost_decision["top_quality_without_cost_evidence"])
print(
    f"Punchline: {excluded} matches the best quality score yet cannot win -- "
    "it reported no cost evidence, and unknown cost is never treated as zero."
)

## 4. Connected comparison checklist

Before any billable request:

1. Require two configured logical model names and resolve both with `context.providers.model(...)`; fail on missing capabilities first.
2. Load one exact prompt version inside every traced prediction.
3. Use the same ordered cases, scorer versions, judge model, inference parameters, and conservative evaluation concurrency.
4. Read latency, input/output tokens, `trace.info.cost`, and cost coverage from completed traces. Account separately for target predictions and judge calls; MLflow evaluation may make an additional prediction while validating tracing.
5. Treat unavailable cost as `None`, not zero. Prefer gateway or billing records for chargeback.
6. Apply absolute quality, policy, critical-row, latency, token, and cost gates. Record a recommendation; do not provision endpoints or mutate deployment aliases from this notebook.

The optional persistence cell records the actual synthetic case records as a Unity Catalog EvaluationDataset and the simulated comparison as a described result run. It deliberately creates no model calls, prompts, or traces.

In [ ]:
from examples.support.cost_quality import persist_cost_quality_evidence

PERSIST_EVIDENCE_TO_DATABRICKS = False
if PERSIST_EVIDENCE_TO_DATABRICKS:
    print(persist_cost_quality_evidence(comparison))
else:
    print("DATABRICKS EVIDENCE PERSISTENCE SKIPPED")